# Neural Network Exercise

In this Exercise Notebook you will be building your own artificial neural network and seeing how adding different types of layers can affect the validation/testing accuracy. Check the NeuralNetworks_Tutorial_part1.ipynb notebook for references

In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle

In [2]:
#Checks if dataset is in working directory, else downloads it
if not os.path.exists("spoken_digit_manual_features.csv"):
    os.system('curl -o spoken_digit_manual_features.csv https://raw.githubusercontent.com/BeaverWorksMedlytics2020/Data_Public/master/NotebookExampleData/Week2/spoken_digit_manual_features.csv')

## Load Training Data and Pre-processed Features

Your goal is to build a neural network that learns to classify which of the 5 speakers is recorded in a signal sample. Your prediction will be based off of features we've already pre-extracted for you and put into this CSV: spectral centroid `SC`, spectral flatness `SF`, and maximum frequency `MF`.

In [3]:
# Load csv containing raw data, labels, and pre-processed features
spoken_df = pd.read_csv('spoken_digit_manual_features.csv', index_col = 0)
print(spoken_df.head(10))
print('\n')

# Set speakers
speakers = set(spoken_df['speaker'])
print(f'There are {len(speakers)} unique speakers in the dataset')

                file  digit   speaker  trial           SC        SF  \
0   5_yweweler_8.wav      5  yweweler      8  1029.497959  0.397336   
1    3_george_49.wav      3    george      4  1881.296834  0.387050   
2  9_yweweler_44.wav      9  yweweler      4  1093.951856  0.394981   
3  8_yweweler_33.wav      8  yweweler      3  1409.543285  0.487496   
4      7_theo_34.wav      7      theo      3   887.361601  0.396825   
5   1_jackson_45.wav      1   jackson      4  1007.568129  0.324100   
6  6_yweweler_18.wav      6  yweweler      1  1286.701352  0.498813   
7    9_george_35.wav      9    george      3  1405.092061  0.353083   
8   9_jackson_32.wav      9   jackson      3  1172.899961  0.477907   
9    8_george_26.wav      8    george      2  1959.977577  0.462901   

           MF  
0  745.878340  
1  323.943662  
2  244.648318  
3  392.350401  
4  130.640309  
5  216.306156  
6  400.715564  
7  447.239693  
8  114.892780  
9  320.537966  


There are 5 unique speakers in the datas

Converting speaker labels to class index labels

In [4]:
# Make dictionary to convert from speaker names to indices
name2int_dict = {name: ind for (ind, name) in enumerate(set(spoken_df['speaker']))}

y_labels = spoken_df['speaker']
# Set y_labels to be indices of speaker
y_labels = [name2int_dict[name] for name in y_labels]

Standardize data and split into train, validation, and test sets:

In [5]:
# Downselect to only the 3 columns of the dataset we are learning from, aka the features
X_data = spoken_df[['SC', 'SF', 'MF']].to_numpy()

# Decide how large to make validation and test sets
n_val = 250
n_test = 250

# Shuffle data before partitioning
X_data, y_labels = shuffle(X_data, y_labels, random_state = 25)

# Partition
X_test, y_test = X_data[:n_test,:], y_labels[:n_test]
X_val, y_val = X_data[n_test:n_test+n_val,:], y_labels[n_test:n_test+n_val]
X_train, y_train = X_data[n_test+n_val:,:], y_labels[n_test+n_val:]

# Scale data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [6]:
import torch

EPOCHS = 50
batch_size = 100

train = torch.utils.data.TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
train_loader = torch.utils.data.DataLoader(train, batch_size=batch_size, shuffle=False)

## Aditional Layers

Before you get to writing your own neural network we'll show you some examples of additional layers you can potetially add that we didn't go over in the tutorial. After reading over our explanations/example code and going through documentation you'll be testing some of these out by putting together a neural network yourself.

### Dropout Layers

During training, drop out layers will randomly zero inputs with probability *p* for each element in the feature dimension.

Empirically, this has proven to help reduce overfitting and improve generalization in neural networks. The common argument is it discourages the network from relying on one feature too heavily.

Theoretically, the reasons for why dropout works isn't well understood--a Stanford paper from 2013 proves mathematically that it is "first order equivalent to L2 regularization." See [here](https://arxiv.org/pdf/1307.1493)

https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html

```python
m = nn.Dropout(p=0.2)
input = torch.randn(20, 16)
output = m(input)
```

### Activation Layers/Functions

Activation functions introduces non-linearity into our network, allowing it to model non-linear functions (instead of just being a linear function).
See [here](https://docs.pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity) for a full list of what is available.

Canonically, sigmoid is the most well-known activation function, but ReLU and its variants is much more popular in practice.

Some popular ones include:
- ReLU
- LeakyReLU
- PReLU
- GeLU
- ELU

There's a bajillion relu variants and they keep making more--when in doubt just go with relu.

```python
# Example
m = nn.ReLU()
input = torch.randn(2)
output = m(input)
```

### Optimization Functions

Optimization functions
- Adam
  - Adam is computationally efficient, has little memory requirement, and is well suited for problems that are large in terms of data/parameter.
- Adagrad
  - Adagrad is an optimizer that is best used for sparse data. Some of its benefits are that it converges more quickly and doesn't need manual adjustment of the hyperparameter "learning rate".
- SGD
  - SGD is a stochastic gradient descent and momentum optimizer. SGD essentially helps gradient vectors move down loss functions towards the minimum point, leading to faster "converging".
- RMSprop
  - As you may already know, the learning rate regulates how much the model
can change based on the estimated error (which occurs every time the model's weights are updated). Instead of treating the learning rate as a hyperparamter, RMSprop is an optimization technique that uses relies on a changing, adaptive learning rate.

See [here](https://docs.pytorch.org/docs/stable/optim.htmlhttps://docs.pytorch.org/docs/stable/optim.html) for a full list and more detailed documentation

```python
# Example code
optimizer = optim.Adam(model.parameters(), lr=0.01)
```

## Putting Together Your Neural Network

Now you will experiment with training different neural networks. We've built a trainer function `train_my_network` for your convenience.

Define your very own neural network, pass the arguments into the function, and see how it does.

Experiment and see what leads to bette/worse performance.

In [7]:
import torch.nn as nn
import torch.optim as optim
def train_my_network(model, optimizer, epochs=50, batch_size=100):
    train = torch.utils.data.TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    )
    train_loader = torch.utils.data.DataLoader(train, batch_size=batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        for i, data in enumerate(train_loader):
            # Train loop
            model.train()
            inputs, labels = data

            # Pytorch keeps track of gradients across loops, so we need to zero it for each batch
            # We need to manually zero the gradient--reset it--every batch
            optimizer.zero_grad()

            outputs = model(inputs) # Get the model's output
            loss = criterion(outputs, labels) # Compute the loss
            preds = torch.argmax(outputs,dim=1)

            # For monitoring
            tr_loss = loss.item()
            tr_acc = torch.sum(preds==labels)/len(labels)

            loss.backward() # Backpropagate to compute gradient
            optimizer.step() # Update the parameters, according to chosen optimizer algorithm
        if epoch%10 == 0:
            print(f'Epoch: {epoch}| Train loss: {tr_loss:.3f}, Train accuracy: {tr_acc: .2f}')

    # Now we put our model through validation set.
    model.eval()
    inputs, labels = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long)

    outputs = model(inputs)
    preds = torch.argmax(outputs,dim=1)

    val_loss = criterion(outputs, labels).item()
    val_acc = torch.sum(preds==labels)/len(labels)
    print(f'Val loss: {tr_loss:.3f}, Val accuracy: {tr_acc: .2f}')

    # return val_loss, val_acc


### Example

In [8]:
example_model = nn.Sequential(
    nn.Linear(3, 8),
    nn.ReLU(),

    *[nn.Linear(8, 8),
    nn.ReLU()]*2,

    nn.Linear(8, 5),
)
optimizer = optim.Adam(example_model.parameters(), lr=0.01)

train_my_network(example_model, optimizer)


Epoch: 0| Train loss: 1.598, Train accuracy:  0.37
Epoch: 10| Train loss: 1.076, Train accuracy:  0.54
Epoch: 20| Train loss: 1.067, Train accuracy:  0.58
Epoch: 30| Train loss: 1.029, Train accuracy:  0.59
Epoch: 40| Train loss: 1.023, Train accuracy:  0.60
Val loss: 1.020, Val accuracy:  0.63


### Model 1

In [9]:
# TODO: experiment with making your own model
model1 = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16,16), nn.ReLU(), nn.Linear(16,5))

optimizer1 = optim.Adam(model1.parameters(), lr = 0.05)
train_my_network(model1, optimizer1)

Epoch: 0| Train loss: 1.101, Train accuracy:  0.52
Epoch: 10| Train loss: 0.939, Train accuracy:  0.58
Epoch: 20| Train loss: 0.903, Train accuracy:  0.65
Epoch: 30| Train loss: 0.835, Train accuracy:  0.60
Epoch: 40| Train loss: 0.825, Train accuracy:  0.62
Val loss: 0.822, Val accuracy:  0.62


### Model 2

In [27]:
# TODO: experiment with making your own model
model2 = nn.Sequential(nn.Linear(3, 32),
                       nn.ReLU(),
                       nn.Dropout(0.1),

                       *[nn.Linear(32,32),
                       nn.ReLU(),
                       nn.Dropout(0.1)]*2,

                       nn.Linear(32,5))

optimizer2 = optim.Adam(model2.parameters(), lr = 0.01)
train_my_network(model2, optimizer2)

Epoch: 0| Train loss: 1.210, Train accuracy:  0.49
Epoch: 10| Train loss: 1.087, Train accuracy:  0.45
Epoch: 20| Train loss: 0.931, Train accuracy:  0.58
Epoch: 30| Train loss: 0.893, Train accuracy:  0.61
Epoch: 40| Train loss: 0.906, Train accuracy:  0.60
Val loss: 0.898, Val accuracy:  0.62


### Model 3

In [46]:
# TODO: experiment with making your own model
model3 = nn.Sequential(nn.Linear(3,64),
                       nn.ReLU(),

                       *[nn.Linear(64, 64),
                         nn.ReLU()]*2,
                        nn.Linear(64,5)
                         )
optimizer3 = optim.Adagrad(model3.parameters(), lr = 0.05)
train_my_network(model3, optimizer3)

Epoch: 0| Train loss: 1.153, Train accuracy:  0.44
Epoch: 10| Train loss: 0.942, Train accuracy:  0.56
Epoch: 20| Train loss: 0.827, Train accuracy:  0.61
Epoch: 30| Train loss: 0.772, Train accuracy:  0.64
Epoch: 40| Train loss: 0.716, Train accuracy:  0.66
Val loss: 0.676, Val accuracy:  0.67


### Once you're done, finalize your model choice and put it through the test set for the final evaluation

In [44]:
def test_my_model(model):
    criterion = nn.CrossEntropyLoss()

    model.eval()
    inputs, labels = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long)

    outputs = model(inputs)
    preds = torch.argmax(outputs,dim=1)

    test_loss = criterion(outputs, labels).item()
    test_acc = torch.sum(preds==labels)/len(labels)

    print(f'Test loss: {test_loss:.2f}, Test accuracy: {test_acc:.2f}')

In [45]:
# TODO: put in your final model
test_my_model(model1)
# TODO: put in your final model
test_my_model(model2)
# TODO: put in your final model
test_my_model(model3)

Test loss: 0.84, Test accuracy: 0.67
Test loss: 0.86, Test accuracy: 0.65
Test loss: 0.79, Test accuracy: 0.70
